## Minimization of the loss function

After the definition of `Ph_data` objects, `Ph_consistency` objects and the loss function `ph_loss` (all contained in `ph_refine.py` with corresponding notebook `ph_refine.ipynb`), let's minimize this loss function.

Notice we have renamed `ph_loss` as `ph_tilde_loss` to stress that it is not computed as the original definition $1/2\chi^2 + \mathrm{reg}$, rather including the minimization of the Gamma function; the `ph_loss` name is then used for defining the loss function defined at the beginning (before the introduction of Gamma)

In [1]:
import numpy as np

In [2]:
from ph_refine import load_ph_data, ph_gamma, ph_tilde_loss, ph_tilde_loss_and_grad

#### 1. load data and evaluate `ph_gamma` and `ph_loss`
the evaluation of `ph_loss` without $\vec\lambda$ fixed requires a while due to the inner minimization of $\tilde\Gamma$

In [3]:
ph_vals = {'A3mer' : [3.00, 3.50, 4.00], 'A5mer' : [3.50, 4.00, 4.50]}

data = load_ph_data(path='Simulation-data', mol_name='A5mer', obs_names=['chi', 'eRMSD'],
                 ph_vals=ph_vals['A5mer'], ref_ph=4.50, g_exp=None, sigma_exp=None)


In [4]:
alphas = np.ones(len(data.g_exp))
ph_weights = np.vstack([data.pops[k] for k in data.pops.keys()])
lambdas = np.ones(len(data.g_exp))
exp_values = np.vstack((data.g_exp, data.sigma_exp))

args = (lambdas, data.legend_matrix, data.gs, exp_values.T, data.p0s, alphas, ph_weights)

ph_gamma(*args)

(Array(5.94410697, dtype=float64),
 Array([1.11661429, 1.45532574], dtype=float64),
 [Array([-0.58452639, -1.80951991, -1.34894436, ..., -0.84716311,
         -0.41223641, -0.50194662], dtype=float64),
  Array([-1.51657242, -0.56728817, -2.8114989 , ..., -1.53192192,
         -0.16676047, -2.87485265], dtype=float64)])

In [5]:
from ph_refine import Manage_indices

table_lambdas = Manage_indices.flat_to_matrix(lambdas, data.legend_matrix)
table_lambdas = np.nan_to_num(table_lambdas)  # put nan to zero

print(table_lambdas.shape)
print(ph_weights.shape)
print(data.gs[1].shape)

j = 0

np.einsum('ki,i,tk->t', table_lambdas, ph_weights[:, j], data.gs[j])

(2, 3)
(3, 2)
(289236, 2)


array([-0.58452639, -1.80951991, -1.34894436, ..., -0.84716311,
       -0.41223641, -0.50194662])

In [6]:
log_pi_ref = np.log(data.pops[data.ref_ph])
log_pi_vec = +log_pi_ref

ph_tilde_loss(log_pi_vec, data)

Array(55.8743632, dtype=float64)

it takes a while due to the inner minimization of $\tilde\Gamma$, without it the function runs fast

In [7]:
from ph_refine import Lambdas

In [8]:
lambdas = Lambdas(np.zeros(len(data.g_exp)), True)

lambdas.value

array([0., 0., 0., 0., 0., 0.])

In [9]:
ph_tilde_loss(log_pi_vec, data, lambdas=lambdas)


Array(1.70267352, dtype=float64)

#### 2. now, minimize `ph_loss`

need to take the optimal value of lambdas, determined by the inner minimization of the Gamma function (inner to `ph_loss`)

one good way to do this should be through a class `Lambdas` (using `global lambdas` is discouraged)

In [10]:
vars(lambdas)

{'value': array([0., 0., 0., 0., 0., 0.]), 'is_fixed': True}

In [11]:
lambdas = Lambdas(np.zeros(len(data.g_exp)), False)

In [12]:
ph_tilde_loss(log_pi_vec, data, lambdas)

Array(55.8743632, dtype=float64)

In [13]:
vars(lambdas)

{'value': array([ -177.62394998,   329.56229979,  -152.30862145,  3903.96499542,
        -7237.01259758,  3333.03669435]),
 'is_fixed': False}

In [14]:
ph_tilde_loss(log_pi_vec, data, lambdas)

Array(55.8743632, dtype=float64)

In [15]:
import jax

In [16]:
ph_tilde_loss_gradient_fun = jax.grad(ph_tilde_loss, argnums=0)

In [17]:
lambdas.is_fixed = True

ph_tilde_loss_gradient_fun(log_pi_vec, data, lambdas)

Array([ 29.49679298, -29.49679298], dtype=float64)

In [18]:
def ph_tilde_loss_and_grad(log_pi_vec, data, lambdas):

    assert not lambdas.is_fixed, 'error: lambdas is fixed'
    loss = ph_tilde_loss(log_pi_vec, data, lambdas)

    lambdas.is_fixed = True
    # in this way, the gradient is computed without looking at the derivative of lambdas w.r.t. log_pi_vec,
    # which is zero, since we are at the optimal lambdas for that log_pi_vec value
    
    grad = ph_tilde_loss_gradient_fun(log_pi_vec, data, lambdas)

    lambdas.is_fixed = False

    print(loss, grad)

    return loss, grad

In [19]:
lambdas = Lambdas(np.zeros(len(data.g_exp)), False)

ph_tilde_loss_and_grad(log_pi_vec, data, lambdas)

55.87436319857597 [ 29.49679298 -29.49679298]


(Array(55.8743632, dtype=float64),
 Array([ 29.49679298, -29.49679298], dtype=float64))

In [20]:
vars(lambdas)

{'value': array([ -177.62394998,   329.56229979,  -152.30862145,  3903.96499542,
        -7237.01259758,  3333.03669435]),
 'is_fixed': False}

In [21]:
ph_tilde_loss_and_grad(log_pi_vec, data, lambdas)

55.87436319857597 [ 29.49679298 -29.49679298]


(Array(55.8743632, dtype=float64),
 Array([ 29.49679298, -29.49679298], dtype=float64))

In [22]:
from scipy.optimize import minimize

In [84]:
lambdas = Lambdas(np.zeros(len(data.g_exp)), False)
args = (data, lambdas)

out = minimize(ph_tilde_loss_and_grad, log_pi_vec, args=args, method='BFGS', jac=True)



55.87436319857597 [ 29.49679298 -29.49679298]
33.85204400336749 [ 4.45613287 -4.45613287]
33.01181566078389 [ 2.25069799 -2.25069799]
32.66367759223028 [ 0.50392341 -0.50392341]
32.64241010465675 [ 0.06859866 -0.06859866]
32.64191075456761 [ 0.01560513 -0.01560513]
32.64200836775343 [-0.016415  0.016415]
32.64197535006769 [-0.00403733  0.00403733]
32.64197431327751 [-0.00321056  0.00321056]
32.64197846976856 [-0.0009069  0.0009069]
32.64197846976794 [-0.00090704  0.00090704]
32.642008355775324 [-0.01619699  0.01619699]
32.641977649335395 [-0.00533416  0.00533416]
32.64197623262641 [-0.00413769  0.00413769]
32.64197569804521 [-0.00347341  0.00347341]
32.64197550066008 [-0.00254474  0.00254474]
32.641979349735244 [-0.00057643  0.00057643]
32.641979343493276 [-0.00024142  0.00024142]
32.64197933244341 [-0.00184707  0.00184707]
32.641979323188 [-0.00235238  0.00235238]
32.64197931777932 [-0.00272097  0.00272097]
32.64197931581267 [-0.00265607  0.00265607]
32.641979314147754 [-0.00284606  0

In [85]:
out

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 32.64191075456761
        x: [-1.371e+00 -1.896e-01]
      nit: 5
      jac: [ 1.561e-02 -1.561e-02]
 hess_inv: [[ 5.556e-01  4.444e-01]
            [ 4.444e-01  5.556e-01]]
     nfev: 23
     njev: 11

In [ ]:
log_pi_vec

array([-0.35672622, -1.20385318])

In [46]:

pi_vec = np.exp(log_pi_vec)
pi_vec /= np.sum(pi_vec)

pi_vec

array([0.69996411, 0.30003589])

In [44]:
out.x

array([-1.37100997, -0.18956943])

In [86]:
x = out.x

x = np.exp(x)
x /= np.sum(x)

x

array([0.23479328, 0.76520672])

#### 3. check minimization
is it true that the loss function has been minimized?

let's define a function `ph_loss` for the loss function ($1/2 \chi^2 + \mathrm{reg}$) and verify we have minimized it

In [4]:
from ph_refine import compute_dkl, compute_ph_weights, Manage_indices, Lambdas
from bussilab import coretools

In [5]:
class Ph_result(coretools.Result):
    def __init__(self, gamma, check_gamma, log_ps, avs, avs_ph, rel_diff, chi2, dkl_p, dkl_pi, loss):
        """ Class with the results of `ph_loss`. """
        
        super().__init__()

        self.gamma = gamma

        self.check_gamma = check_gamma

        self.log_ps = log_ps

        self.avs = avs

        self.avs_ph = avs_ph

        self.rel_diff = rel_diff

        self.chi2 = chi2

        self.dkl_p = dkl_p
        
        self.dkl_pi = dkl_pi

        self.loss = loss

In [6]:
def ph_loss(lambdas, pis, data, alphas = 1, alpha_pi = 1):
    """
    Function that computes the loss function for pH refinement, defined as in documentation (in short,
    1/2 chi2 + reg. terms).

    """

    if alphas is float or int:
        assert alphas == 1, 'error on alphas, it should be an array!'
        alphas = np.ones(len(data.ns_prot))
    
    exp_values = np.vstack((data.g_exp, data.sigma_exp)).T

    ph_weights = []

    for i in range(len(data.ph_vals)):
        w = compute_ph_weights(np.log(pis), data.log_fugacities[i], data.ns_prot)
        ph_weights.append(w)

    ph_weights = np.array(ph_weights)

    gamma, logZs, corrections = ph_gamma(lambdas, data.legend_matrix, data.gs, exp_values, data.p0s, alphas, ph_weights)

    log_ps = []
    avs = []
    avs_correction = []

    for j in range(len(data.ns_prot)):

        log_ps.append(np.log(data.p0s[j]) - logZs[j] - corrections[j])

        p = np.exp(log_ps[j])
        avs.append(np.dot(p, data.gs[j]))
        avs_correction.append(np.dot(p, corrections[j]))

    avs = np.vstack(avs)  # (N. prot. states x N. obs) matrix

    avs_ph = np.dot(ph_weights, avs).T
    # avs_ph is a full matrix (N. obs x N. pH), but it may happen that only some of its cells have a
    # corresponding experimental value

    exp_vals = Manage_indices.flat_to_matrix(data.g_exp, data.legend_matrix)
    exp_errs = Manage_indices.flat_to_matrix(data.sigma_exp, data.legend_matrix)

    rel_diff = np.where(np.isnan(exp_vals) | np.isnan (exp_errs), np.nan, (avs_ph - exp_vals)/exp_errs)

    chi2 = np.sum(rel_diff**2)

    loss = 1/2*chi2

    dkl_p = []

    for j in range(len(data.ns_prot)):
        dkl_p.append(-logZs[j] - avs_correction[j])

    dkl_pi = compute_dkl(pis, data.pops[data.ref_ph])

    loss += np.dot(alphas, dkl_p) + alpha_pi*dkl_pi

    check_gamma = 1/2*chi2 + np.dot(alphas, dkl_p) + gamma

    return Ph_result(gamma, check_gamma, log_ps, avs, avs_ph, rel_diff, chi2, dkl_p, dkl_pi, loss)

In [7]:
pi_vec = data.pops[data.ref_ph]
lambdas_vec = np.zeros(len(data.g_exp))

print(pi_vec)

out = ph_loss(lambdas=lambdas_vec, pis=pi_vec, data=data)#, alphas=np.ones(len(data.ns_prot)), alpha_pi=1)

[0.69996411 0.30003589]


/tmp/ipykernel_18968/82421268.py:30: RuntimeWarning: divide by zero encountered in log
  log_ps.append(np.log(data.p0s[j]) - logZs[j] - corrections[j])


In [8]:
out

         avs: array([[-1.54506178,  1.08278492],
                     [-0.06397881,  1.35580914]])
      avs_ph: array([[-0.34414487, -0.69275702, -1.10068373],
                     [ 1.30416306,  1.23989957,  1.16470198]])
 check_gamma: Array(29039.33324039, dtype=float64)
        chi2: 58078.66648078074
       dkl_p: [Array(-0., dtype=float64), Array(-2.22044605e-16, dtype=float64)]
      dkl_pi: Array(0., dtype=float64)
       gamma: Array(2.22044605e-16, dtype=float64)
      log_ps: [Array([-17.95436422, -17.66668215, -16.20912477, ..., -13.82089607,
                     -14.62867439, -14.26094961], dtype=float64), Array([-17.47963824, -18.1909128 , -16.62382244, ..., -13.7973674 ,
                     -14.61146719, -14.04545789], dtype=float64)]
        loss: Array(29039.33324039, dtype=float64)
    rel_diff: array([[ -50.88746977, -112.22398641, -183.45089532],
                     [   4.63868002,  -49.61151337,  -82.20619301]])

In [9]:
from scipy.optimize import minimize

In [10]:
lambdas = Lambdas(np.zeros(len(data.g_exp)), False)
args = (data, lambdas)
log_pi_vec = np.log(pi_vec)

mini = minimize(ph_tilde_loss_and_grad, log_pi_vec, args=args, method='BFGS', jac=True)


-54.17168967641427 0.0
Traced<ConcreteArray(-54.17168967641427, dtype=float64)>with<JVPTrace(level=2/0)> with
  primal = Array(-54.17168968, dtype=float64)
  tangent = Traced<ShapedArray(float64[])>with<JaxprTrace(level=1/0)> with
    pval = (ShapedArray(float64[]), None)
    recipe = JaxprEqnRecipe(eqn_id=<object object at 0x7fb18050f050>, in_tracers=(Traced<ShapedArray(float64[]):JaxprTrace(level=1/0)>,), out_tracer_refs=[<weakref at 0x7fb180511590; to 'JaxprTracer' at 0x7fb1805114a0>], out_avals=[ShapedArray(float64[])], primitive=pjit, params={'jaxpr': { lambda ; a:f64[]. let  in (a,) }, 'in_shardings': (UnspecifiedValue,), 'out_shardings': (UnspecifiedValue,), 'in_layouts': (None,), 'out_layouts': (None,), 'resource_env': None, 'donated_invars': (False,), 'name': 'add', 'keep_unused': False, 'inline': True}, effects=set(), source_info=SourceInfo(traceback=<jaxlib.xla_extension.Traceback object at 0x559d929eaab0>, name_stack=NameStack(stack=(Transform(name='jvp'),))), ctx=JaxprEqnC

In [11]:
mini

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 29.27627718490753
        x: [-1.787e+00  2.263e-01]
      nit: 8
      jac: [-4.161e-04  4.161e-04]
 hess_inv: [[ 1.080e+00 -7.971e-02]
            [-7.971e-02  1.080e+00]]
     nfev: 27
     njev: 15

In [12]:
lambdas_new = lambdas.value

pi_new = np.exp(mini.x)
print(np.sum(pi_new))

pi_new

1.4214606218749182


array([0.16747786, 1.25398276])

In [13]:
ph_loss(lambdas_new, pi_new, data)#, alphas=np.ones(len(data.ns_prot)), alpha_pi=1)

/tmp/ipykernel_18968/82421268.py:30: RuntimeWarning: divide by zero encountered in log
  log_ps.append(np.log(data.p0s[j]) - logZs[j] - corrections[j])


         avs: array([[-1.11207242,  0.94284939],
                     [-0.10467443,  1.30692033]])
      avs_ph: array([[-0.11795159, -0.14549711, -0.22336704],
                     [ 1.302122  ,  1.29216712,  1.26402514]])
 check_gamma: Array(2.95029248e-06, dtype=float64)
        chi2: 56.66819808390798
       dkl_p: [Array(0.17563596, dtype=float64), Array(0.02506225, dtype=float64)]
      dkl_pi: Array(1.55389525, dtype=float64)
       gamma: Array(-28.53479431, dtype=float64)
      log_ps: [Array([-17.36381074, -17.48875313, -16.07984416, ..., -14.78974885,
                     -15.5231716 , -15.15169798], dtype=float64), Array([-17.041346  , -17.90600057, -15.98036985, ..., -14.02722229,
                     -14.81036703, -14.40183795], dtype=float64)]
        loss: Array(30.08869251, dtype=float64)
    rel_diff: array([[-3.32420465,  4.45055373, -1.20524439],
                     [ 2.851467  , -3.88189297,  1.07603995]])

In [14]:
vars(lambdas)

{'value': array([ -698.8688273 ,   949.07000085,  -250.54148316,  2495.45700567,
        -3395.7537753 ,   903.01587223]),
 'is_fixed': False}

In [15]:
lambdas.is_fixed = True

ph_tilde_loss(np.log(pi_new), data, lambdas)

-28.534794305782917 0.7414830406219095


Array(29.27627735, dtype=float64)

In [16]:
lambdas.is_fixed = False

ph_tilde_loss(np.log(pi_new), data, lambdas)

-28.534794305782917 0.7414830406219095


Array(29.27627735, dtype=float64)

In [17]:
out = ph_loss(lambdas.value, pi_new, data)

out

/tmp/ipykernel_18968/82421268.py:30: RuntimeWarning: divide by zero encountered in log
  log_ps.append(np.log(data.p0s[j]) - logZs[j] - corrections[j])


         avs: array([[-1.11207242,  0.94284939],
                     [-0.10467443,  1.30692033]])
      avs_ph: array([[-0.11795159, -0.14549711, -0.22336704],
                     [ 1.302122  ,  1.29216712,  1.26402514]])
 check_gamma: Array(2.95029248e-06, dtype=float64)
        chi2: 56.66819808390798
       dkl_p: [Array(0.17563596, dtype=float64), Array(0.02506225, dtype=float64)]
      dkl_pi: Array(1.55389525, dtype=float64)
       gamma: Array(-28.53479431, dtype=float64)
      log_ps: [Array([-17.36381074, -17.48875313, -16.07984416, ..., -14.78974885,
                     -15.5231716 , -15.15169798], dtype=float64), Array([-17.041346  , -17.90600057, -15.98036985, ..., -14.02722229,
                     -14.81036703, -14.40183795], dtype=float64)]
        loss: Array(30.08869251, dtype=float64)
    rel_diff: array([[-3.32420465,  4.45055373, -1.20524439],
                     [ 2.851467  , -3.88189297,  1.07603995]])

In [18]:
dkl = np.sum(pi_new*(np.log(pi_new) - np.log(data.pops[data.ref_ph])))

dkl

1.5538952516404896

In [19]:
log_pi_new = np.log(pi_new)
log_pi_new -= np.mean(log_pi_new)

log_pi_ref = np.log(data.pops[data.ref_ph])

np.sum(np.exp(log_pi_new)*(log_pi_new - log_pi_ref))

5.811044910770843

In [ ]:
np.sum(np.exp(log_pi_vec)*(log_pi_vec - log_pi_ref))